# GPU-Solver Colab Watch (Git-우편함)
터널 없음. cmd/result JSON을 git repo로 비동기 교환.
설계: GPU-Solver/docs/03-git-mailbox-runner.md

**실측 watch**: REQ 받으면 `executor.execute_request`로 gate(reference 교차검증)+ncu 프로파일 실행 → signal_dict RES 반환.

**선결**: Colab Secrets(🔑 사이드바)에 `GPU_MAILBOX_TOKEN` 등록 (fine-grained PAT, gpu-mailbox+gpu-solver-loop Contents R/W). 셀2가 `userdata.get`으로 읽음 — 평문 토큰 0.


In [7]:
# 1) GPU 확인 (A100 선택했는지)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-fa2b4507-645d-1ef2-89e9-548c18ec57f7)


In [ ]:
# 2) git 인증 — Colab Secrets (평문 토큰 0). 🔑 사이드바에 GPU_MAILBOX_TOKEN 등록 필요.
import subprocess
from google.colab import userdata

PAT = userdata.get("GPU_MAILBOX_TOKEN")   # 암호화 저장 — 셀 출력·git 이력에 안 남음
USER = "alexxony"
EMAIL = "colab@gpu-solver"

subprocess.run(["git", "config", "--global", "user.email", EMAIL])
subprocess.run(["git", "config", "--global", "user.name", "colab-watch"])

def clone(repo):
    url = "https://%s:%s@github.com/%s/%s.git" % (USER, PAT, USER, repo)
    subprocess.run(["rm", "-rf", "/content/" + repo])
    r = subprocess.run(["git", "clone", "-q", url, "/content/" + repo],
                       capture_output=True, text=True)
    print(repo, "->", "OK" if r.returncode == 0 else r.stderr)

clone("gpu-mailbox")
clone("gpu-solver-loop")


gpu-mailbox -> remote: Invalid username or token. Password authentication is not supported for Git operations.
fatal: Authentication failed for 'https://github.com/alexxony/gpu-mailbox.git/'

gpu-solver-loop -> remote: Invalid username or token. Password authentication is not supported for Git operations.
fatal: Authentication failed for 'https://github.com/alexxony/gpu-solver-loop.git/'



In [12]:
# 3) watch.py 임포트 경로 추가 + 골격 self-check (GPU 없이 로직 확인)
import sys
sys.path.insert(0, '/content/gpu-solver-loop')
!cd /content/gpu-solver-loop/loop && python watch.py --selfcheck

watch.py self-check PASS


In [ ]:
import sys
sys.path.insert(0, "/content/gpu-solver-loop/loop")
import watch

MB = "/content/gpu-mailbox"
print("watch 시작 — runner REQ 대기 (무한).")
watch.watch_loop(MB, poll_s=5.0, max_iters=None)   # 무한 — 한 번 띄우면 계속. 끝낼 때만 수동 정지(■).
print("watch 종료. result/ 확인:")
!ls -la /content/gpu-mailbox/result/


In [ ]:
import subprocess
r = subprocess.run(["git","-C","/content/gpu-mailbox","pull","--no-edit"],
                   capture_output=True, text=True)
print("STDOUT:", r.stdout)
print("STDERR:", r.stderr)
print("RC:", r.returncode)